In [1]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import polars as pl

from ethos.constants import PROJECT_ROOT
from ethos.constants import SpecialToken as ST
from ethos.inference.constants import Reason, Task
from ethos.metrics import compute_and_print_metrics, preprocess_inference_results

results_dir = PROJECT_ROOT / "results"

## ICU Admission results

In [ ]:
def process_icu_admission_results(input_dir, **kwargs) -> pl.DataFrame:
    return preprocess_inference_results(
        input_dir,
        actual_expr=pl.col("actual").is_in([ST.ICU_ADMISSION]),
        expected_expr=pl.col("expected").is_in([ST.ICU_ADMISSION]),
        filter_ambiguous=pl.col("stop_reason") == Reason.GOT_TOKEN,
        **kwargs,
    )

In [2]:
# DEBUGGING ONLY
import sys
from pathlib import Path

project_root = Path.cwd()
while not (project_root / 'pyproject.toml').exists() and project_root != project_root.parent:
    project_root = project_root.parent

sys.path.insert(0, str(project_root))
from src.ethos.datasets.mimic_icu import ICUAdmissionDataset
path = "../../../../mnt/data_share/project_henri/ethos-ares/mimic-tokenized/test"
dataset = ICUAdmissionDataset(path)
sum_adm = sum([True if d[1]['expected'] == 'ICU_ADMISSION' else False for d in dataset])
sum_adm#/len(dataset)

64594 64594


9956

In [3]:
import json
true_ids = [d[1]['hadm_id'] for d in dataset if d[1]['expected'] == 'ICU_ADMISSION']
with open("true_ids.json", "w") as f:
    json.dump(true_ids, f)


In [7]:
for i in range(len(dataset)): 
    if dataset[i][1]['hadm_id'] == 27768240.0:
        print(i, dataset[i][1])
        break

17624 {'expected': 'HOSPITAL_DISCHARGE', 'true_token_dist': 1, 'true_token_time': 1000000, 'icu_stay_id': None, 'patient_id': 18193001, 'hadm_id': 27768240, 'prediction_time': 5103973440000000, 'data_idx': 14777930}


In [4]:
import pandas as pd
df_token = pd.read_parquet(f"{path}/11_inject_time_intervals/1.parquet")

In [6]:
df_token[df_token.hadm_id == 27768240.0].iloc[:10]

,subject_id,time,code,numeric_value,text_value,hadm_id,edstay_id,icustay_id,insurance,language,marital_status,race,drg_severity,drg_mortality,emar_id,emar_seq,priority
14777927,18193001,2131-09-27 18:24:00,18h-1d,NaN,URGENT,27768240.0,NaN,NaN,Other,ENGLISH,MARRIED,WHITE,NaN,NaN,NaN,NaN,NaN
14777928,18193001,2131-09-27 18:24:00,HOSPITAL_ADMISSION,NaN,URGENT,27768240.0,NaN,NaN,Other,ENGLISH,MARRIED,WHITE,NaN,NaN,NaN,NaN,NaN
14777929,18193001,2131-09-27 18:24:00,ADMISSION_TYPE//EMERGENCY,NaN,URGENT,27768240.0,NaN,NaN,Other,ENGLISH,MARRIED,WHITE,NaN,NaN,NaN,NaN,NaN
14777930,18193001,2131-09-27 18:24:00,INSURANCE//OTHER,NaN,URGENT,27768240.0,NaN,NaN,Other,ENGLISH,MARRIED,WHITE,NaN,NaN,NaN,NaN,NaN
14777939,18193001,2131-09-27 18:25:51,TRANSFER//CARDIAC_VASCULAR_INTENSIVE_CARE_UNIT...,NaN,NaN,27768240.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14777940,18193001,2131-09-27 18:25:51,ICU_ADMISSION,NaN,ICU_TYPE//Cardiac Vascular Intensive Care Unit...,27768240.0,NaN,35458898.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14777941,18193001,2131-09-27 18:25:51,ICU_TYPE//CARDIAC_VASCULAR_INTENSIVE_CARE_UNIT...,NaN,ICU_TYPE//Cardiac Vascular Intensive Care Unit...,27768240.0,NaN,35458898.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14777942,18193001,2131-09-27 18:25:51,SOFA,NaN,ICU_TYPE//Cardiac Vascular Intensive Care Unit...,27768240.0,NaN,35458898.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14777943,18193001,2131-09-27 18:25:51,Q1,NaN,ICU_TYPE//Cardiac Vascular Intensive Care Unit...,27768240.0,NaN,35458898.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14777944,18193001,2131-09-27 18:44:00,15m-45m,1.0,NaN,27768240.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,STAT


In [ ]:
df_21472555 = df_token[df_token['hadm_id'].isin([23747233,25248373])]
df_21472555[df_21472555.code.str.contains('HOSPITAL') | df_21472555.code.str.contains('ICU_') | df_21472555.code.str.contains('DEATH')]

In [ ]:
df_token.iloc[11579570:11579580]

In [ ]:
df2 = df_token[df_token['hadm_id'] == 21430309.0]
df2[df2.code.str.contains('DRG')]
# df_token.iloc[11573392+66]

In [ ]:
icu_admission_results = []
for input_dir in (results_dir / Task.ICU_ADMISSION).iterdir():
    df = process_icu_admission_results(input_dir)
    rep_num = df["counts"].mean()
    res = compute_and_print_metrics(
        df["expected"],
        df["actual"],
        f"ICU Admission (rep_num={rep_num:.2f})\n{input_dir.name}",
    )
    icu_admission_results.append(
        {
            "name": input_dir.name,
            "auc": res["fitted_auc"],
            "rep_num": rep_num,
        }
    )
    plt.show()
icu_admission_results = pl.DataFrame(icu_admission_results).sort("auc", descending=True)

In [ ]:
icu_admission_results